transaction_manager.py
------------------------
Implements the required flow:

    User -> Python PoA validation -> APPROVE/REJECT -> Ethereum -> event
         -> Python blockchain audit record

Regulatory/financial validators require UNANIMOUS approval — there is no
partial-majority "threshold". A vote that is not unanimous does NOT
auto-reject the transaction: it moves to UNDER_INSPECTION in the mempool
(mempool.py) and waits for a compliance reviewer to call
resolve_inspection(). Only a unanimous REJECT at vote time, or an
inspection explicitly resolved as rejected, produces a final REJECTED
transaction.
The Python chain never claims to own tokens;
it only records decisions and, after Ethereum confirms, the real
on-chain result.

# 1. Libraries

Note that we will need all other code files to run this one , Blockchain.py,mempool.py,validator.py,ethereum_client.py to send to interact with etherum blockchain to to send the trx and to recieve the tx hash of the etherum block to be audited on the private blockchain, event_listner.py

In [ ]:
import time
import uuid

import config
from blockchain import Blockchain, PoAValidator
from ethereum_client import EthereumClient
from event_listener import extract_event
from mempool import Mempool, UNDER_INSPECTION
from validator import ValidatorNode

# 2. TransactionManager Class

In [ ]:
class TransactionManager:
    def __init__(self, chain: Blockchain, recorder: PoAValidator, eth: EthereumClient):
        """
        chain    : the existing Python PoA blockchain (audit ledger)
        recorder : the validator whose signature seals each AUDIT block
        eth      : EthereumClient for reading state + sending approved txs
        """
        self.chain = chain
        self.recorder = recorder
        self.eth = eth
        self.validators = [ValidatorNode(vid, PoAValidator(vid)) for vid in config.VALIDATOR_IDS]
        for v in self.validators:
            self.chain.register_validator(v.keypair)
        self.mempool = Mempool()

    # ------------------------------------------------------------
    def submit(self, op_type: str, params: dict, forced_votes: dict = None) -> dict:
        """
        forced_votes: optional {validator_id: (approve_bool, reason_str)}
        for scripting an exact vote pattern in a demo scenario. Everything
        downstream of the vote — unanimity check, signing, Ethereum send
        gating, mempool state, audit recording — always runs for real.
        """
        forced_votes = forced_votes or {}
        tx_id = str(uuid.uuid4())
        operation = {"tx_id": tx_id, "op_type": op_type, "params": params}

        print(f"\n>>> Transaction proposed: {op_type} (tx_id={tx_id[:8]}...)")
        self.mempool.submit(tx_id, op_type, params)
        self.chain.add_transaction({
            "event": "TRANSACTION_PROPOSED",
            "tx_id": tx_id, "op_type": op_type, "params": params,
            "timestamp": time.time(),
        })

        # ---- 1. independent validator votes ----
        decisions = []
        for v in self.validators:
            if v.validator_id in forced_votes:
                f_approve, f_reason = forced_votes[v.validator_id]
                decision = v.evaluate(operation, self.eth, forced_approve=f_approve, forced_reason=f_reason)
            else:
                decision = v.evaluate(operation, self.eth)
            label = "APPROVE" if decision.approve else "REJECT"
            print(f"    {v.validator_id} -> {label}  ({decision.reason})")
            decisions.append(decision)

        self.mempool.record_votes(tx_id, decisions)
        approvals = [d for d in decisions if d.approve]
        rejections = [d for d in decisions if not d.approve]
        unanimous_approval = len(rejections) == 0
        unanimous_rejection = len(approvals) == 0

        audit_common = {
            "tx_id": tx_id, "op_type": op_type,
            "validator_decisions": [d.to_dict() for d in decisions],
            "approvals_received": len(approvals),
            "total_validators": len(self.validators),
            "timestamp": time.time(),
        }

        # ---- 2a. unanimous approval -> execute ----
        if unanimous_approval:
            print(f"    {len(approvals)}/{len(self.validators)} approvals — UNANIMOUS.")
            self.mempool.mark_approved(tx_id)
            self.chain.add_transaction({**audit_common, "event": "TRANSACTION_APPROVED"})
            block = self.chain.mine_block(self.recorder)
            print(f"    Python PoA audit block created (#{block.index}).")
            return self._execute_on_ethereum(tx_id, op_type, params)

        # ---- 2b. unanimous rejection -> final REJECTED ----
        if unanimous_rejection:
            print(f"    0/{len(self.validators)} approvals — UNANIMOUS REJECTION. Transaction REJECTED.")
            self.mempool.mark_rejected(tx_id, "unanimous rejection by all validators")
            self.chain.add_transaction({**audit_common, "event": "TRANSACTION_REJECTED",
                                         "reason": "unanimous rejection"})
            block = self.chain.mine_block(self.recorder)
            print(f"    Python PoA audit block created (#{block.index}) — no Ethereum tx sent.")
            return {"status": "REJECTED", "approved": False, "tx_id": tx_id, "audit_block": block.index}

        # ---- 2c. split decision -> UNDER_INSPECTION (not auto-rejected) ----
        objectors = [d.validator_id for d in rejections]
        reason = f"{len(rejections)} validator(s) objected: {objectors}"
        print(f"    {len(approvals)}/{len(self.validators)} approvals — NOT UNANIMOUS.")
        print(f"    Transaction placed UNDER INSPECTION (manual/compliance review required).")
        self.mempool.mark_under_inspection(tx_id, reason)
        self.chain.add_transaction({**audit_common, "event": "TRANSACTION_UNDER_INSPECTION", "reason": reason})
        block = self.chain.mine_block(self.recorder)
        print(f"    Python PoA audit block created (#{block.index}) — held in mempool, no Ethereum tx sent.")
        return {"status": "UNDER_INSPECTION", "approved": False, "tx_id": tx_id, "audit_block": block.index}

    # ------------------------------------------------------------
    def resolve_inspection(self, tx_id: str, resolver_id: str, decision: bool, justification: str) -> dict:
        """
        Manually resolves a transaction sitting in UNDER_INSPECTION.
        decision=True  -> approved by override, proceeds to Ethereum
        decision=False -> rejected, final
        Either way the resolution + justification is written to the
        audit chain, signed like any other block.
        """
        record = self.mempool.get(tx_id)
        if record["status"] != UNDER_INSPECTION:
            raise ValueError(f"tx {tx_id} is not under inspection (status={record['status']})")

        print(f"\n>>> Compliance resolution for tx {tx_id[:8]}... by {resolver_id}")
        print(f"    Decision: {'APPROVE (override)' if decision else 'REJECT (final)'}")
        print(f"    Justification: {justification}")

        self.chain.add_transaction({
            "event": "TRANSACTION_INSPECTION_RESOLVED",
            "tx_id": tx_id, "resolver": resolver_id,
            "decision": "APPROVED" if decision else "REJECTED",
            "justification": justification, "timestamp": time.time(),
        })
        block = self.chain.mine_block(self.recorder)
        print(f"    Python PoA audit block created (#{block.index}).")

        if decision:
            self.mempool.mark_approved(tx_id, note=f"inspection resolved as approved by {resolver_id}")
            return self._execute_on_ethereum(tx_id, record["op_type"], record["params"])

        self.mempool.mark_rejected(tx_id, f"inspection resolved as rejected by {resolver_id}: {justification}")
        return {"status": "REJECTED", "approved": False, "tx_id": tx_id, "audit_block": block.index}

    # ------------------------------------------------------------
    def _execute_on_ethereum(self, tx_id: str, op_type: str, params: dict) -> dict:
        print("    Sending transaction to Ethereum...")
        send_fn = getattr(self.eth, f"send_{op_type}")
        receipt = send_fn(params)
        eth_tx_hash = receipt.transactionHash.hex()
        print(f"    Ethereum TX: {eth_tx_hash}")
        print(f"    Transaction confirmed in Ethereum block {receipt.blockNumber}")

        event_info = extract_event(self.eth.vault, op_type, receipt)
        print(f"    Smart contract event detected: {event_info['event_name']} {event_info['args']}")

        confirmed_event_name = {
            "create_property": "PROPERTY_CREATED",
            "transfer_shares": "SHARE_TRANSFERRED",
            "purchase_hedge": "HEDGE_PURCHASED",
            "settle_hedge": "HEDGE_SETTLED",
        }[op_type]

        self.chain.add_transaction({
            "event": confirmed_event_name,
            "tx_id": tx_id,
            "property_id": params.get("property_id"),
            "sender": params.get("from_addr") or params.get("holder") or params.get("caller"),
            "recipient": params.get("to_addr"),
            "params": params,
            "ethereum_tx_hash": eth_tx_hash,
            "ethereum_block_number": receipt.blockNumber,
            "contract_address": receipt.to,
            "solidity_event": event_info,
            "ipfs_cid": params.get("ipfs_cid"),
            "timestamp": time.time(),
        })
        confirm_block = self.chain.mine_block(self.recorder)
        print(f"    Python PoA audit block created (#{confirm_block.index}) — confirmed result recorded.")

        self.mempool.mark_confirmed(tx_id, eth_tx_hash)

        return {
            "status": "CONFIRMED", "approved": True, "tx_id": tx_id,
            "eth_tx_hash": eth_tx_hash, "audit_block": confirm_block.index,
            "event": event_info,
        }